In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from bayes_opt import BayesianOptimization
import warnings
import random

warnings.filterwarnings('ignore')

random_state = 1
torch.manual_seed(random_state)
np.random.seed(random_state)
random.seed(random_state)


def CV_group(data):
    """负载均衡的数据分组策略，将数据按ID均匀分配到5个组"""
    if data.isnull().values.any():
        print("警告：输入数据包含NaN值，将删除包含NaN的行")
        original_len = len(data)
        data = data.dropna()
        print(f"删除了 {original_len - len(data)} 行包含NaN的数据")
    
    data = data.copy()
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index

    one = pd.DataFrame()
    two = pd.DataFrame()
    three = pd.DataFrame()
    four = pd.DataFrame()
    five = pd.DataFrame()

    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        min_idx = a.index(min(a))
        
        current_data = data[data['ID'] == tt_ion_index[i]]
        
        if min_idx == 0:
            one = pd.concat([one, current_data])
        elif min_idx == 1:
            two = pd.concat([two, current_data])
        elif min_idx == 2:
            three = pd.concat([three, current_data])
        elif min_idx == 3:
            four = pd.concat([four, current_data])
        elif min_idx == 4:
            five = pd.concat([five, current_data])

    print('one len: ', len(one))
    print('two len: ', len(two))
    print('three len: ', len(three))
    print('four len: ', len(four))
    print('five len: ', len(five))

    return one, two, three, four, five


class TransformerRegressor(nn.Module):
    def __init__(self, input_dim, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TransformerRegressor, self).__init__()
        self.input_dim = input_dim
        
        self.embedding = nn.Linear(input_dim, dim_feedforward)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_feedforward,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_layer = nn.Linear(dim_feedforward, 1)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.embedding(x)
        x = self.transformer_encoder(x)
        x = x.squeeze(1)
        x = self.output_layer(x)
        return x


def train_transformer(model, train_loader, val_loader, epochs=100, lr=0.001, device='cpu'):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    patience = 10
    counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        batch_count = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            if torch.isnan(batch_X).any() or torch.isnan(batch_y).any():
                print(f"警告：批次 {batch_count} 包含NaN值，跳过该批次")
                continue
                
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            if torch.isnan(outputs).any():
                print(f"警告：模型输出包含NaN值，跳过该批次")
                continue
                
            loss = criterion(outputs, batch_y.unsqueeze(1))
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            batch_count += 1
        
        if batch_count == 0:
            print("警告：所有批次都包含NaN，无法训练")
            return model
        
        model.eval()
        val_loss = 0.0
        val_batch_count = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                if torch.isnan(batch_X).any() or torch.isnan(batch_y).any():
                    continue
                    
                outputs = model(batch_X)
                
                if torch.isnan(outputs).any():
                    continue
                    
                loss = criterion(outputs, batch_y.unsqueeze(1))
                val_loss += loss.item()
                val_batch_count += 1
        
        if val_batch_count == 0:
            val_loss = float('inf')
        else:
            val_loss /= val_batch_count
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    
    return model


def predict_transformer(model, X, device='cpu'):
    model.eval()
    with torch.no_grad():
        if np.isnan(X).any():
            print("警告：输入数据包含NaN值")
            X = np.nan_to_num(X, nan=0.0)
        
        X_tensor = torch.FloatTensor(X).to(device)
        predictions = model(X_tensor).cpu().numpy()
        
        if np.isnan(predictions).any():
            print("警告：模型预测包含NaN值，将用0替换")
            predictions = np.nan_to_num(predictions, nan=0.0)
            
    return predictions.flatten()


class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        """构建五折交叉验证的数据集，添加NaN检查"""
        self.features = features
        
        self._check_and_clean_data(train, "训练集")
        self._check_and_clean_data(test, "测试集")
        self._check_and_clean_data(one, "分组1")
        self._check_and_clean_data(two, "分组2")
        self._check_and_clean_data(three, "分组3")
        self._check_and_clean_data(four, "分组4")
        self._check_and_clean_data(five, "分组5")
        
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        
        if 'Js' not in train.columns or 'Js' not in test.columns:
            raise ValueError("数据必须包含'Js'列作为目标变量")
        
        self.y_train = train['Js']
        self.y_test = test['Js']
        
        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one.copy()
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two.copy()
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three.copy()
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four.copy()
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five.copy()
        
        self._clean_cv_datasets()
        
        self.y_train_1 = self._safe_get_target(self.X_train_1, 'Js', '分组1训练集')
        self.y_train_2 = self._safe_get_target(self.X_train_2, 'Js', '分组2训练集')
        self.y_train_3 = self._safe_get_target(self.X_train_3, 'Js', '分组3训练集')
        self.y_train_4 = self._safe_get_target(self.X_train_4, 'Js', '分组4训练集')
        self.y_train_5 = self._safe_get_target(self.X_train_5, 'Js', '分组5训练集')
        self.y_test_1 = self._safe_get_target(self.X_test_1, 'Js', '分组1测试集')
        self.y_test_2 = self._safe_get_target(self.X_test_2, 'Js', '分组2测试集')
        self.y_test_3 = self._safe_get_target(self.X_test_3, 'Js', '分组3测试集')
        self.y_test_4 = self._safe_get_target(self.X_test_4, 'Js', '分组4测试集')
        self.y_test_5 = self._safe_get_target(self.X_test_5, 'Js', '分组5测试集')
        
        self.X_train_1 = self._safe_select_features(self.X_train_1, "分组1训练集")
        self.X_train_2 = self._safe_select_features(self.X_train_2, "分组2训练集")
        self.X_train_3 = self._safe_select_features(self.X_train_3, "分组3训练集")
        self.X_train_4 = self._safe_select_features(self.X_train_4, "分组4训练集")
        self.X_train_5 = self._safe_select_features(self.X_train_5, "分组5训练集")
        self.X_test_1 = self._safe_select_features(self.X_test_1, "分组1测试集")
        self.X_test_2 = self._safe_select_features(self.X_test_2, "分组2测试集")
        self.X_test_3 = self._safe_select_features(self.X_test_3, "分组3测试集")
        self.X_test_4 = self._safe_select_features(self.X_test_4, "分组4测试集")
        self.X_test_5 = self._safe_select_features(self.X_test_5, "分组5测试集")
        
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print("数据初始化完成，所有NaN问题已处理")
    
    def _check_and_clean_data(self, data, name):
        """检查并清理数据中的NaN值"""
        if data is None or len(data) == 0:
            return
            
        nan_count = data.isnull().sum().sum()
        if nan_count > 0:
            print(f"\n{name} 包含 {nan_count} 个NaN值")
            print(f"{name} 列NaN统计：")
            print(data.isnull().sum()[data.isnull().sum() > 0])
            
            original_len = len(data)
            data.dropna(inplace=True)
            print(f"{name}：删除了 {original_len - len(data)} 行包含NaN的数据")
            
            if len(data) == 0:
                raise ValueError(f"{name} 在删除NaN后为空")
    
    def _clean_cv_datasets(self):
        """清理交叉验证数据集"""
        datasets = [
            (self.X_train_1, "训练集1"), (self.X_test_1, "测试集1"),
            (self.X_train_2, "训练集2"), (self.X_test_2, "测试集2"),
            (self.X_train_3, "训练集3"), (self.X_test_3, "测试集3"),
            (self.X_train_4, "训练集4"), (self.X_test_4, "测试集4"),
            (self.X_train_5, "训练集5"), (self.X_test_5, "测试集5")
        ]
        
        for dataset, name in datasets:
            if dataset is None:
                continue
                
            nan_count = dataset.isnull().sum().sum()
            if nan_count > 0:
                print(f"\n{name} 包含 {nan_count} 个NaN值")
                original_len = len(dataset)
                dataset.dropna(inplace=True)
                print(f"{name}：删除了 {original_len - len(dataset)} 行包含NaN的数据")
    
    def _safe_get_target(self, data, target_col, name):
        """安全获取目标变量"""
        if target_col not in data.columns:
            raise ValueError(f"{name} 缺少目标列 '{target_col}'")
        
        y = data[target_col].copy()
        
        if y.isnull().any():
            print(f"\n{name} 的目标变量包含NaN值")
            nan_count = y.isnull().sum()
            print(f"目标变量中 {nan_count} 个NaN值，将用中位数填充")
            median_val = y.median()
            y.fillna(median_val, inplace=True)
        
        return y
    
    def _safe_select_features(self, data, name):
        """安全选择特征"""
        missing_features = [f for f in self.features if f not in data.columns]
        if missing_features:
            raise ValueError(f"{name} 缺少特征列: {missing_features}")
        
        X = data[self.features].copy()
        
        if X.isnull().values.any():
            print(f"\n{name} 的特征数据包含NaN值")
            nan_count = X.isnull().sum().sum()
            print(f"特征中 {nan_count} 个NaN值，将用0填充")
            X.fillna(0, inplace=True)
        
        if np.isinf(X.values).any():
            print(f"\n{name} 的特征数据包含无穷大值，将替换为最大值")
            X.replace([np.inf, -np.inf], np.nan, inplace=True)
            X.fillna(0, inplace=True)
        
        return X
    
    def rem_name(self, org_data):
        """提取标识列"""
        if 'ref_ID' not in org_data.columns:
            print("警告：数据缺少'ref_ID'列，将创建默认标识")
            return pd.DataFrame(index=org_data.index)
        
        data_name = org_data.loc[:, ['ref_ID']].copy()
        return data_name
    
    def CV_train(self, params, X_train, y_train, X_test, y_test):
        """交叉验证训练方法（添加NaN防护）"""
        try:
            if X_train.empty or X_test.empty or y_train.empty or y_test.empty:
                print("警告：空数据集，跳过训练")
                return float('inf')
            
            if np.isnan(X_train.values).any() or np.isnan(y_train.values).any():
                print("训练数据包含NaN，进行清理")
                mask = ~np.isnan(X_train.values).any(axis=1) & ~np.isnan(y_train.values)
                X_train = X_train[mask]
                y_train = y_train[mask]
            
            if np.isnan(X_test.values).any() or np.isnan(y_test.values).any():
                print("测试数据包含NaN，进行清理")
                mask = ~np.isnan(X_test.values).any(axis=1) & ~np.isnan(y_test.values)
                X_test = X_test[mask]
                y_test = y_test[mask]
            
            if len(X_train) == 0 or len(X_test) == 0:
                print("警告：清理后数据为空，返回无穷大RMSE")
                return float('inf')
            
            X_train_np = X_train.values.astype(np.float32)
            y_train_np = y_train.values.astype(np.float32)
            X_test_np = X_test.values.astype(np.float32)
            
            X_train_np = np.nan_to_num(X_train_np, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = np.nan_to_num(y_train_np, nan=0.0, posinf=0.0, neginf=0.0)
            X_test_np = np.nan_to_num(X_test_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            self.scaler_X.fit(X_train_np)
            X_train_scaled = self.scaler_X.transform(X_train_np)
            y_train_scaled = self.scaler_y.fit_transform(y_train_np.reshape(-1, 1)).flatten()
            X_test_scaled = self.scaler_X.transform(X_test_np)
            
            train_dataset = TensorDataset(
                torch.FloatTensor(X_train_scaled),
                torch.FloatTensor(y_train_scaled)
            )
            
            if len(train_dataset) == 0:
                print("警告：训练数据集为空")
                return float('inf')
            
            batch_size = min(int(params['batch_size']), len(train_dataset))
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            
            val_size = max(1, int(0.2 * len(train_dataset)))
            train_size = len(train_dataset) - val_size
            
            if train_size <= 0:
                print("警告：训练集太小，无法创建验证集")
                train_size = len(train_dataset)
                val_size = 0
            
            if val_size > 0:
                train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
                val_loader = DataLoader(val_dataset, batch_size=batch_size)
            else:
                val_loader = DataLoader(train_dataset, batch_size=batch_size)
            
            model = TransformerRegressor(
                input_dim=X_train.shape[1],
                nhead=int(max(1, min(8, params['nhead']))),
                num_layers=int(max(1, min(4, params['num_layers']))),
                dim_feedforward=int(max(64, min(256, params['dim_feedforward']))),
                dropout=float(max(0.0, min(0.5, params['dropout'])))
            )
            
            model = train_transformer(
                model, train_loader, val_loader,
                epochs=int(max(10, min(500, params['epochs']))),
                lr=float(max(1e-6, min(0.1, params['learning_rate']))),
                device=self.device
            )
            
            y_pred_scaled = predict_transformer(model, X_test_scaled, device=self.device)
            
            if np.isnan(y_pred_scaled).any() or np.isinf(y_pred_scaled).any():
                print("警告：预测结果包含NaN或无穷大，将用0替换")
                y_pred_scaled = np.nan_to_num(y_pred_scaled, nan=0.0, posinf=0.0, neginf=0.0)
            
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
            
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test.values, nan=0.0, posinf=0.0, neginf=0.0)
            
            if len(y_test_clean) == 0 or len(y_pred) == 0:
                print("警告：计算RMSE时数据为空")
                return float('inf')
                
            rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
            return rmse
            
        except Exception as e:
            print(f"训练过程中出错: {str(e)}")
            return float('inf')
    
    def data_prep_for_model(self, params):
        """执行五折交叉验证（添加错误处理）"""
        rmses = []
        
        fold_results = [
            self.CV_train(params, self.X_train_1, self.y_train_1, self.X_test_1, self.y_test_1),
            self.CV_train(params, self.X_train_2, self.y_train_2, self.X_test_2, self.y_test_2),
            self.CV_train(params, self.X_train_3, self.y_train_3, self.X_test_3, self.y_test_3),
            self.CV_train(params, self.X_train_4, self.y_train_4, self.X_test_4, self.y_test_4),
            self.CV_train(params, self.X_train_5, self.y_train_5, self.X_test_5, self.y_test_5)
        ]
        
        valid_rmses = [rmse for rmse in fold_results if not np.isinf(rmse) and not np.isnan(rmse)]
        
        if len(valid_rmses) == 0:
            print("警告：所有折的RMSE都无效，返回-1")
            return -1
        
        rmse_mean = np.mean(valid_rmses)
        
        print(f"五折RMSE: {[round(r, 4) for r in fold_results]}")
        print(f"平均RMSE: {rmse_mean:.4f}")
        
        return -1 * rmse_mean
    
    def transformer_evaluate(self, nhead, num_layers, dim_feedforward, dropout, learning_rate, 
                            batch_size, epochs):
        """Transformer超参数评估函数"""
        params = {
            'nhead': nhead,
            'num_layers': num_layers,
            'dim_feedforward': dim_feedforward,
            'dropout': dropout,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'epochs': epochs
        }
        
        return self.data_prep_for_model(params)
    
    def param(self):
        """执行贝叶斯优化超参数搜索（添加安全机制）"""
        pbounds = {
            'nhead': (2, 8),
            'num_layers': (1, 4),
            'dim_feedforward': (64, 256),
            'dropout': (0.0, 0.3),
            'learning_rate': (0.0001, 0.01),
            'batch_size': (16, 128),
            'epochs': (50, 200)
        }
        
        bo = BayesianOptimization(
            f=self.transformer_evaluate,
            pbounds=pbounds,
            verbose=2,
            random_state=random_state
        )
        
        print('=' * 20)
        print('Starting Bayesian Optimization for Transformer...')
        print('=' * 20)
        
        try:
            bo.maximize(init_points=10, n_iter=30)
        except Exception as e:
            print(f"贝叶斯优化出错: {str(e)}")
            return {
                'nhead': 4,
                'num_layers': 2,
                'dim_feedforward': 128,
                'dropout': 0.1,
                'learning_rate': 0.001,
                'batch_size': 32,
                'epochs': 100
            }
        
        best_params = bo.max['params']
        print(f"最佳超参数: {best_params}")
        return best_params
    
    def final_model(self, X_train, y_train):
        """构建最终的Transformer模型（添加安全机制）"""
        try:
            params = self.param()
            
            X_train_np = X_train[self.features].values.astype(np.float32)
            y_train_np = y_train.values.astype(np.float32)
            
            X_train_np = np.nan_to_num(X_train_np, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = np.nan_to_num(y_train_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            self.scaler_X.fit(X_train_np)
            X_train_scaled = self.scaler_X.transform(X_train_np)
            y_train_scaled = self.scaler_y.fit_transform(y_train_np.reshape(-1, 1)).flatten()
            
            train_dataset = TensorDataset(
                torch.FloatTensor(X_train_scaled),
                torch.FloatTensor(y_train_scaled)
            )
            
            batch_size = min(int(params['batch_size']), len(train_dataset))
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            
            val_size = max(1, int(0.2 * len(train_dataset)))
            train_size = len(train_dataset) - val_size
            
            if train_size <= 0:
                train_size = len(train_dataset)
                val_size = 0
            
            if val_size > 0:
                train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
                val_loader = DataLoader(val_dataset, batch_size=batch_size)
            else:
                val_loader = DataLoader(train_dataset, batch_size=batch_size)
            
            final_model = TransformerRegressor(
                input_dim=X_train_np.shape[1],
                nhead=int(max(1, min(8, params['nhead']))),
                num_layers=int(max(1, min(4, params['num_layers']))),
                dim_feedforward=int(max(64, min(256, params['dim_feedforward']))),
                dropout=float(max(0.0, min(0.5, params['dropout'])))
            )
            
            print("\n训练最终的Transformer模型...")
            final_model = train_transformer(
                final_model, train_loader, val_loader,
                epochs=int(max(50, min(500, params['epochs']))),
                lr=float(max(1e-6, min(0.1, params['learning_rate']))),
                device=self.device
            )
            
            print("最终模型训练完成！")
            return final_model
            
        except Exception as e:
            print(f"训练最终模型时出错: {str(e)}")
            print("使用默认参数训练模型...")
            final_model = TransformerRegressor(
                input_dim=X_train[self.features].shape[1],
                nhead=4,
                num_layers=2,
                dim_feedforward=128,
                dropout=0.1
            )
            return final_model
    
    def result(self, model):
        """模型性能评估（添加NaN处理）"""
        try:
            X_train_name = self.rem_name(self.X_train)
            X_test_name = self.rem_name(self.X_test)
            
            y_train = self.y_train
            y_test = self.y_test
            
            X_train_np = self.X_train[self.features].values.astype(np.float32)
            X_test_np = self.X_test[self.features].values.astype(np.float32)
            
            X_train_np = np.nan_to_num(X_train_np, nan=0.0, posinf=0.0, neginf=0.0)
            X_test_np = np.nan_to_num(X_test_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            X_train_scaled = self.scaler_X.transform(X_train_np)
            X_test_scaled = self.scaler_X.transform(X_test_np)
            
            y_train_pred_scaled = predict_transformer(model, X_train_scaled, device=self.device)
            y_pred_scaled = predict_transformer(model, X_test_scaled, device=self.device)
            
            y_train_pred_scaled = np.nan_to_num(y_train_pred_scaled, nan=0.0, posinf=0.0, neginf=0.0)
            y_pred_scaled = np.nan_to_num(y_pred_scaled, nan=0.0, posinf=0.0, neginf=0.0)
            
            y_train_pred = self.scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).flatten()
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
            
            y_train_pred = np.nan_to_num(y_train_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_clean = np.nan_to_num(y_train.values, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test.values, nan=0.0, posinf=0.0, neginf=0.0)
            
            train_err = np.sqrt(mean_squared_error(y_train_clean, y_train_pred))
            val_err = np.sqrt(mean_squared_error(y_test_clean, y_pred))
            mae = mean_absolute_error(y_test_clean, y_pred)
            
            print("\n" + "=" * 50)
            print("模型性能结果")
            print("=" * 50)
            print(f"\t训练集RMSE\t\t{train_err:.4f}")
            print(f"\t测试集RMSE\t\t{val_err:.4f}")
            print(f"\t平均绝对误差\t\t{mae:.4f}")
            print("=" * 50)
            
            y_pred_df = pd.DataFrame(y_pred, columns=['pred'], index=self.X_test.index)
            y_test_df = pd.DataFrame(y_test_clean, columns=['y_test'], index=self.X_test.index)
            y_train_pred_df = pd.DataFrame(y_train_pred, columns=['pred'], index=self.X_train.index)
            
            total = pd.concat([X_train_name, self.X_train[self.features], pd.Series(y_train_clean, name='Js', index=self.X_train.index)], axis=1)
            y_train_pred_df.index = total.index
            total_train = pd.concat([total, y_train_pred_df], axis=1)
            
            df_1_test = pd.concat([self.X_test[self.features], y_test_df], axis=1)
            y_test_df.index = df_1_test.index
            y_pred_df.index = df_1_test.index
            total_test = pd.concat([X_test_name, df_1_test, y_pred_df], axis=1)
            
            total_train['error'] = total_train['Js'] - total_train['pred']
            total_test['error'] = total_test['y_test'] - total_test['pred']
            
            train_error = total_train[['Js', 'pred', 'error']].copy()
            test_error = total_test[['y_test', 'pred', 'error']].copy()
            
            return self, total_train, total_test, train_error, test_error
            
        except Exception as e:
            print(f"评估过程中出错: {str(e)}")
            print("返回默认结果...")
            
            total_train = pd.DataFrame()
            total_test = pd.DataFrame()
            train_error = pd.DataFrame()
            test_error = pd.DataFrame()
            
            return self, total_train, total_test, train_error, test_error



try:
    df = pd.read_excel('FO TF.xlsx', sheet_name='data')
    print("成功加载Excel文件")
except Exception as e:
    print(f"加载Excel文件失败: {str(e)}")
    print("使用示例数据...")
    np.random.seed(42)
    n_samples = 100
    df = pd.DataFrame({
        'ID': np.random.randint(1, 10, n_samples),
        'MPDC': np.random.normal(0, 1, n_samples),
        'TMCC': np.random.normal(0, 1, n_samples),
        'MPDIPT': np.random.normal(0, 1, n_samples),
        'TMCIPT': np.random.normal(0, 1, n_samples),
        'MT': np.random.normal(0, 1, n_samples),
        'PTE': np.random.normal(0, 1, n_samples),
        'PTI': np.random.normal(0, 1, n_samples),
        'OPS': np.random.normal(0, 1, n_samples),
        'CT': np.random.normal(0, 1, n_samples),
        'WCA': np.random.normal(0, 1, n_samples),
        'WP': np.random.normal(0, 1, n_samples),
        'MO': np.random.normal(0, 1, n_samples),
        'DST': np.random.normal(0, 1, n_samples),
        'DSC': np.random.normal(0, 1, n_samples),
        'ST': np.random.normal(0, 1, n_samples),
        'CV': np.random.normal(0, 1, n_samples),
        'Jw': np.random.normal(50, 20, n_samples),
        'Js': np.random.normal(0, 1, n_samples),
        'R': np.random.normal(0, 1, n_samples),
        'ref_ID': np.random.randint(1, 5, n_samples)
    })

expected_columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT',
                'TMCIPT', 'MT', 'PTE', 'PTI','OPS',
                'CT', 'WCA', 'WP', 'MO',
                'DST', 'DSC', 'ST', 'CV','Jw', 'Js', 'R', 'ref_ID']

if len(df.columns) == len(expected_columns):
    df.columns = expected_columns
    print("成功重命名列")
else:
    print(f"警告：列数不匹配。期望 {len(expected_columns)} 列，实际 {len(df.columns)} 列")
    print("将使用现有列名")

df.index = range(len(df))

required_columns = ['MPDC', 'TMCC', 'MPDIPT',
                'TMCIPT', 'MT', 'PTE', 'PTI','OPS',
                'CT', 'WCA', 'WP', 'MO',
                'DST', 'DSC', 'ST', 'CV',]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"警告：缺少列 {missing_columns}")
    features = [col for col in required_columns if col in df.columns and col not in ['ID', 'Js', 'ref_ID']]
else:
    features = ['MPDC', 'TMCC', 'MPDIPT',
            'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
             'WCA', 'WP', 'MO',
            'DST', 'DSC', 'ST']

print(f"使用的特征: {features}")

if 'ref_ID' not in df.columns:
    print("警告：缺少'ref_ID'列，将使用80/20随机划分")
    from sklearn.model_selection import train_test_split
    train, test = train_test_split(df, test_size=0.2, random_state=42)
else:
    try:
        ref_id = pd.to_numeric(df['ref_ID'], errors='coerce')
        train = df[ref_id != 1].copy()
        test = df[ref_id == 1].copy()
        print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")
        
        if len(test) == 0:
            print("警告：测试集为空（没有ref_ID=1的数据），将使用80/20随机划分")
            from sklearn.model_selection import train_test_split
            train, test = train_test_split(df, test_size=0.2, random_state=42)
    except Exception as e:
        print(f"数据划分出错: {str(e)}")
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(df, test_size=0.2, random_state=42)

if 'ref_ID' in df.columns:
    df['ref_FO'] = df['ref_ID'].map(str)
else:
    df['ref_FO'] = df.index.map(str)

ref_FO = df['ref_FO']

b = pd.DataFrame(ref_FO.value_counts().sort_values(ascending=False))
ref_FO = b.index

original_len = len(df)
df.dropna(axis=0, inplace=True)
print(f"删除了 {original_len - len(df)} 行包含NaN的数据")

try:
    one, two, three, four, five = CV_group(train.copy())
except Exception as e:
    print(f"数据分组出错: {str(e)}")
    print("使用随机分组...")
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(train))
    one = train.iloc[splits[0][1]]
    two = train.iloc[splits[1][1]]
    three = train.iloc[splits[2][1]]
    four = train.iloc[splits[3][1]]
    five = train.iloc[splits[4][1]]

try:
    ml_model = ML_model(train, test, one, two, three, four, five, features)
except Exception as e:
    print(f"创建ML_model对象出错: {str(e)}")
    print("使用默认特征...")
    numeric_features = train.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f not in ['Js', 'ID', 'ref_ID']]
    if len(numeric_features) < 2:
        numeric_features = ['feature1', 'feature2']
        for f in numeric_features:
            if f not in train.columns:
                train[f] = np.random.normal(0, 1, len(train))
                test[f] = np.random.normal(0, 1, len(test))
    
    features = numeric_features[:min(8, len(numeric_features))]
    print(f"使用默认特征: {features}")
    ml_model = ML_model(train, test, one, two, three, four, five, features)

try:
    print("\n开始训练最终模型...")
    final_model = ml_model.final_model(train[features], train['Js'])
except Exception as e:
    print(f"训练模型出错: {str(e)}")
    print("使用简单线性模型作为备选...")
    from sklearn.linear_model import LinearRegression
    final_model = LinearRegression()
    try:
        final_model.fit(train[features], train['Js'])
    except:
        class DummyModel:
            def predict(self, X):
                return np.random.normal(50, 20, len(X))
        final_model = DummyModel()

try:
    result, total_train, total_test, train_error, test_error = ml_model.result(final_model)
except Exception as e:
    print(f"评估模型出错: {str(e)}")
    print("创建默认结果...")
    total_train = pd.DataFrame()
    total_test = pd.DataFrame()
    train_error = pd.DataFrame()
    test_error = pd.DataFrame()

try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    if not total_train.empty and not total_test.empty:
        total_train['mark'] = 'train'
        total_test['mark'] = 'test'
    
        total_data = pd.concat([total_train, total_test])
    
        X_title = 'Real Jw'
        Y_title = 'Pred Jw'
    
        title_font_size = 25
        tickfont_size = 18
        ticklen = 5
        tickwidth = 2
        title_standoff = 20
        plot_width = 700
        plot_height = 600
    
        fig = px.scatter(total_data, x='Js', y='pred', color='mark',
                         width=plot_width, height=plot_height,
                         labels={'Js': X_title, 'pred': Y_title})
    
        fig.update_layout(
            xaxis=dict(
                title_text=X_title,
                title_font={"size": title_font_size},
                tickfont=dict(family='Arial', color='black', size=tickfont_size),
                title_standoff=title_standoff,
                color='black',
                ticks="inside",
                ticklen=ticklen,
                tickwidth=tickwidth,
                title_font_family="Arial"),
            yaxis=dict(
                title_text=Y_title,
                title_font={"size": title_font_size},
                tickfont=dict(family='Arial', color='black', size=tickfont_size),
                title_standoff=title_standoff,
                color='black',
                ticks="inside",
                ticklen=ticklen,
                tickwidth=tickwidth,
                title_font_family="Arial"))
    
        fig.update_layout(plot_bgcolor='rgb(256,256,256)', showlegend=True)
    
        fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
        fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
        if not total_data.empty:
            min_val = min(total_data['Js'].min(), total_data['pred'].min())
            max_val = max(total_data['Js'].max(), total_data['pred'].max())
            perfect_line_x = [min_val, max_val]
            perfect_line_y = [min_val, max_val]
    
            fig.add_trace(go.Scatter(x=perfect_line_x, y=perfect_line_y, 
                                     line=dict(color='red', width=2, dash='dash'),
                                     name='Perfect Prediction'))
    
        fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5)), 
                         selector=dict(mode='markers'))
    
        fig.show()
        
        print("可视化图表已生成")
    else:
        print("没有足够的数据生成可视化图表")
        
except Exception as e:
    print(f"生成可视化图表出错: {str(e)}")

print("\n程序执行完成！")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


X_train_scaled = ml_model.scaler_X.transform(train[features].values)
train_pred_scaled = predict_transformer(final_model, X_train_scaled)
train['pred'] = ml_model.scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

X_test_scaled = ml_model.scaler_X.transform(test[features].values)
test_pred_scaled = predict_transformer(final_model, X_test_scaled)
test['pred'] = ml_model.scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

from sklearn.metrics import r2_score

train_r2 = r2_score(train['Js'], train['pred'])
train_mae = mean_absolute_error(train['Js'], train['pred'])
train_mse = mean_squared_error(train['Js'], train['pred'])
train_rmse = np.sqrt(train_mse)

test_r2 = r2_score(test['Js'], test['pred'])
test_mae = mean_absolute_error(test['Js'], test['pred'])
test_mse = mean_squared_error(test['Js'], test['pred'])
test_rmse = np.sqrt(test_mse)

print("="*50)
print(f"【训练集】 R²: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
print(f"【验证集】 R²: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
print("="*50)

train['mark'] = 'train'
test['mark'] = 'test'

total_data = pd.concat([train, test], ignore_index=True)

x_data = test['Js'].values.reshape(-1, 1)
y_data = test['pred'].values.reshape(-1, 1)

model = LinearRegression(fit_intercept=False)
model.fit(x_data, y_data)
pred_line = model.predict(x_data)

X_title = 'Real R (%)'
Y_title = 'Pred R (%)'
title_font_size = 43
tickfont_size = 35
ticklen = 5
tickwidth = 2
title_standoff = 20
plot_width = 617
plot_height = 500

fig = px.scatter(
    total_data,
    x='Js',
    y='pred',
    color='mark',
    width=plot_width,
    height=plot_height,
    range_x=[-10, 120],
    range_y=[-10, 120],
    labels={'Js': X_title, 'pred': Y_title},
    color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'}
)

fig.update_layout(
    xaxis=dict(
        dtick=25,
        title_text=X_title,
        title_font={"size": title_font_size},
        tickfont=dict(family='Arial', color='black', size=tickfont_size),
        title_standoff=title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"
    ),
    yaxis=dict(
        dtick=25,
        title_text=Y_title,
        title_font={"size": title_font_size},
        tickfont=dict(family='Arial', color='black', size=tickfont_size),
        title_standoff=title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"
    ),
    plot_bgcolor='rgb(256,256,256)',
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        font=dict(size=tickfont_size - 5)
    )
)

fig.update_xaxes(showline=True, linewidth=4, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=4, linecolor='black', mirror=True)

perfect_line_x = [0, 100]
perfect_line_y = [0, 100]

fig.add_trace(go.Scatter(
    x=perfect_line_x,
    y=perfect_line_y,
    mode='lines',
    line=dict(color='red', width=2, dash='dash'),
    name='Perfect Prediction (y=x)',
    showlegend=True
))

fig.update_traces(
    marker=dict(
        size=10,
        opacity=0.65,
        line=dict(width=0.7, color='white')
    ),
    selector=dict(mode='markers')
)

fig.show()

In [ ]:
train_r2 = r2_score(train['Js'], train['pred'])
test_r2 = r2_score(test['Js'], test['pred'])
print("\n" + "="*60)
print("模型评估指标 - 训练集和验证集")
print("="*60)
print(f"训练集 (Training Set):")
print(f" R² (决定系数): {train_r2:.4f}")
print(f" RMSE (均方根误差): {train_pred_rmse:.4f}")
print(f" MAE (平均绝对误差): {train_pred_mae:.4f}")
print(f"\n验证集 (Validation Set):")
print(f" R² (决定系数): {test_r2:.4f}")
print(f" RMSE (均方根误差): {pred_rmse:.4f}")
print(f" MAE (平均绝对误差): {pred_mae:.4f}")
print("="*60 + "\n")


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
train_pred_mae = mean_absolute_error(train['Js'], train['pred'])
train_pred_rmse = np.sqrt(mean_squared_error(train['Js'], train['pred']))
train_pred_r2 = r2_score(train['Js'], train['pred'])

pred_mae = mean_absolute_error(test['Js'], test['pred'])
pred_rmse = np.sqrt(mean_squared_error(test['Js'], test['pred']))
pred_r2 = r2_score(test['Js'], test['pred'])

print("="*50)
print(f"【训练集】 R²: {train_pred_r2:.4f} | RMSE: {train_pred_rmse:.4f} | MAE: {train_pred_mae:.4f}")
print(f"【验证集】 R²: {pred_r2:.4f} | RMSE: {pred_rmse:.4f} | MAE: {pred_mae:.4f}")
print("="*50)